**Why ingestion exists at all** comes down to one limitation: a model only knows what's in its training data and whatever you put in the prompt — never your internal documents, never today's FD rates, never an SOP written last week.
- This is the exact same gap RAG exists to close — ingestion is just the **first** stage of that pipeline, the part that gets raw files into a shape anything downstream can use
- The "new-product-name problem" is really an ingestion-and-retrieval problem stacked together: the knowledge exists somewhere (a PDF), it's just never been **ingested** into anything the model can search
- Get ingestion wrong, and every later stage — chunking, embedding, retrieval — inherits the damage. Garbage in, garbage out applies here more than almost anywhere else in the pipeline

**The Document object pattern** is the one decision that makes everything downstream source-agnostic.
- Every loader — PDF, text file, CSV — returns the same shape: `page_content` (the actual text) plus `metadata` (a dict describing where it came from)
- Once a `Document` exists, chunking and embedding code never needs an `if it's a PDF / if it's a CSV` branch anywhere — that's the entire point of standardizing on one shape
- `knowledge_base's `{"text":, "source":}` dicts were a stripped-down version of this same idea — this sub-chapter formalizes it into a real `@dataclass`, with richer metadata (page numbers, row indices) that the dict version was throwing away

**Text loading** is the simplest loader here, on purpose — `fd_keyword_groups.txt` and `fd_negation_phrases.txt` are small, structured reference files, not documents that benefit from being split into multiple `Document`s.
- `TextLoader` reads the whole file into **one** `Document` — splitting these into pieces would actively hurt: a keyword group only makes sense as a complete list, not torn into fragments
- This is the same loader that would handle plain-text SOPs or logs if the NBFC ever has them — the loader doesn't care about content, only about "give me everything in this file as one block"

**CSV/Excel loading** treats every row as its own retrievable unit, which is a different granularity decision than the text loader made.
- `CSVLoader` mirrors LangChain's `CSVLoader`: **one `Document` per row**, with every column folded into `page_content` as `key: value` lines, and the row index kept in `metadata`
- Run against the real `fd_master_database.csv`, row 0 becomes a `Document` whose content is Shobha Chopra's entire FD record — that's now something you could chunk, embed, and search individually, not just `pandas`-filter
- This is the loader that would matter if the agent ever needs to **search across customer records semantically** rather than by exact `FD_No` lookup — which is exactly the access-control question flagged for Sub-Chapter 5

**Why drop PDF handling entirely?** 
- Parsing PDFs (tables, scanned pages, OCR, encrypted files) is a genuinely hard, specialized problem — hard enough that an entire team exists just to solve it well

**`load()` vs `lazy_load()`** is a memory decision that's invisible at your current scale and very visible at the NBFC's real scale.
- Load (eager loading) means a resource — data, a module, a model, an object — is loaded immediately when the program starts or when the parent object is created, regardless of whether it's actually used right away.
- Lazy load means the resource is loaded only at the moment it's actually needed (on first access), not before.

#### ML Model Serving (FastAPI / Flask Inference Service)

#### Eager Loading

```python
# Loaded once when the server process starts
model = load_model("bert-large.pt")  # takes 30s, uses 4GB RAM

@app.post("/predict")
def predict(text: str):
    return model(text)
```

- Server takes 30s to become "ready," but every request afterward is fast and predictable.
- Good for production APIs with constant traffic — you want failures (corrupt model file, OOM) caught at deploy time, not mid-request.

#### Lazy Loading

```python
model = None

@app.post("/predict")
def predict(text: str):
    global model
    if model is None:
        model = load_model("bert-large.pt")  # only happens on first call
    return model(text)
```

- Server starts instantly, but the first user to call `/predict` eats a 30-second delay.
- Risky in production (looks like the service is hanging) unless combined with a "warm-up" request right after deploy.

#### Why Do People Use Lazy Loading for ML Models?

If eager loading gives predictable latency, why would anyone choose lazy loading? A few real production reasons:

1. **Fast deployments / fast restarts** — If your CI/CD pipeline redeploys frequently (multiple times a day), eager loading means every deploy waits 30s+ before the health check passes. Lazy loading lets the container report "ready" instantly — important for rolling deployments or frequent pod scaling.

2. **Many models, but each instance only needs a few** — A service hosting 100 customer-specific models: eager-loading all 100 at boot wastes massive memory and startup time on models that pod may never serve. Lazy loading plus caching only loads what's actually requested.

3. **Serverless / scale-to-zero environments** — AWS Lambda, Cloud Run, Knative spin up new instances on demand and kill idle ones; there's no long-lived "boot once, serve forever" process. Eager loading at "startup" still triggers a lazy-style cold start on every fresh instance anyway, so teams often lazy-load deliberately and optimize separately with caching or warm containers.

4. **Memory pressure with many optional model paths** — A service supporting multiple models (e.g., a small fast model and a large accurate model, used conditionally) wastes RAM if all options are eager-loaded. Lazy loading only pays the memory cost for the path actually used.

5. **Faster developer iteration** — During local development, nobody wants to wait 30s every restart to test an unrelated endpoint. Lazy loading speeds up local dev loops.

6. **Avoiding wasted work entirely** — A rarely used model (e.g., a fallback triggered 1% of the time) costs load time on every single restart with eager loading. Lazy loading pays that cost only when actually needed.

#### The Trade-off

> Lazy loading optimizes for startup time and average resource usage, at the cost of unpredictable first-request latency.

#### The Production-Grade Answer (Interview Tip)

Mature systems rarely use pure lazy loading. They combine it with a **warm-up call** right after deploy — hitting `/predict` once internally before accepting real traffic. This gives fast startup like lazy loading, and no user-facing cold start like eager loading.

This hybrid approach is usually the answer interviewers are looking for when asked "lazy vs eager — which do you use?"

**Duplicate documents under different filenames**  — is a two-stage problem, cheapest check first.
- **Stage 1, hash**: `content_hash()` (SHA-256) catches **exact** duplicates instantly — same content, byte for byte, regardless of filename. Computing a hash is nearly free, so this runs on everything
- **Stage 2, cosine similarity**: only computed for documents that **survived** the hash check — embeddings are the expensive step, so paying for them on content you'd already have rejected is wasted cost
- Tested against 4 documents simulating exactly this filename pattern: the byte-identical pair was caught by the hash, and a *reworded* near-duplicate (same fact, different phrasing) was caught by cosine similarity — two different kinds of duplication, two different tools, applied in cost order

**Incremental ingestion** is the same discipline `insert_fd_record()` / `update_fd_record()` already established for the FD database, applied to documents instead of rows.
- A **manifest** (`ingestion_manifest.json`) tracks each source file's content hash from the last time it was ingested
- `get_files_to_ingest()` compares current file hashes against the manifest and returns **only** what's new or changed — everything else gets skipped entirely, no re-chunking, no re-embedding, no wasted API calls
- Tested directly: changed the content of exactly one tracked file, and confirmed only **that** file showed up for re-ingestion — the unchanged file never got touched, which is the entire point and the part most naive "just reload everything" pipelines get wrong